# 09 · Residual Connection ve Layer Normalization

**Belgedeki bölüm:** Residual Connection ve Layer Normalization

- **Residual (kestirme yol):** bloğa giren X, bloğun çıktısına eklenir: `X + MHA(X)`. Ham anlam unutulmaz, gradyanlar derin ağda kaybolmaz.
- **LayerNorm:** her token'ın vektörü kendi içinde ortalama 0, varyans 1 olacak şekilde ölçeklenir.
- İkisi birlikte **Add & Norm**: `LayerNorm(X + MHA(X))`.

In [ ]:
import torch
from usta_embedding import UstaEmbedding
from usta_tokenizer import UstaTokenizer

u_tokenizer = UstaTokenizer("tokenizer.json")

prompt = "the capital of united"

tokens = u_tokenizer.encode(prompt)

# UstaEmbedding = nn.Embedding + RoPE pozisyon bilgisi (04_Positional_Encoding, usta_embedding.py)
torch.manual_seed(1)
u_embedding = UstaEmbedding(vocab_size=len(u_tokenizer.vocab), embedding_dim=4, context_length=32)

sentence_meanings = u_embedding(tokens)
sentence_meanings.shape

### 1 · Residual: X + MHA(X)

In [ ]:
from usta_multi_head_attention import UstaMultiHeadAttention

torch.manual_seed(1)
mha = UstaMultiHeadAttention(embedding_dim=4, output_dim=4, context_length=32, num_heads=2)

mha_out = mha(sentence_meanings)
residual = sentence_meanings + mha_out  # Yeni matris = X + MHA çıktısı

sentence_meanings.shape, mha_out.shape, residual.shape  # boyut değişmez

### Neden? Derin ağda gradyan kaybolması (vanishing gradient)
30 katmanlı küçük bir ağda girdiye geri ulaşan gradyanın büyüklüğünü kestirme yol olmadan ve kestirme yolla karşılaştıralım.

In [ ]:
import torch.nn as nn

def input_gradient(use_residual, depth=30):
  torch.manual_seed(0)
  layers = [nn.Linear(4, 4) for _ in range(depth)]
  x = torch.randn(1, 4, requires_grad=True)
  h = x
  for layer in layers:
    out = torch.tanh(layer(h))
    h = h + out if use_residual else out
  h.sum().backward()
  return x.grad.norm().item()

print("kestirme yol yok :", input_gradient(False))
print("kestirme yol var :", input_gradient(True))

### 2 · Layer Normalization (UstaLayerNorm)

In [ ]:
from usta_layer_norm import UstaLayerNorm

norm_layer = UstaLayerNorm(4)
normalized = norm_layer(residual)
normalized

In [ ]:
# Her satır (token) için: ortalama ≈ 0, varyans ≈ 1
normalized.mean(dim=-1), normalized.var(dim=-1, unbiased=False)

In [ ]:
# PyTorch'un hazır LayerNorm'u ile aynı sonuç
torch.allclose(normalized, nn.LayerNorm(4, elementwise_affine=False)(residual), atol=1e-5)

### 3 · Add & Norm = LayerNorm(X + MHA(X))

In [ ]:
add_and_norm = norm_layer(sentence_meanings + mha(sentence_meanings))
add_and_norm